# 05 — Route context calibration (terrain, buffer, dwell)

Infrastructure-dependent parameters that shape a *route* rather than its price:
terrain, the timetable buffer quota, and minimum dwell. They live here because
they are properties of a country's track infrastructure, and they land in the
same `track_infrastructures` row as the charges. Written to
`data/route_context.csv`; narrative in `TERRAIN_AND_BUFFER.md`.

**Terrain carries two independent metrics.** Cumulative ascent `A` (m/km)
drives energy; ruling gradient `G` (per mille) drives whether a heavy consist
needs a second locomotive. A rolling country climbs constantly but never
steeply; a flat country with one mountain crossing is the reverse.

**Everything here is country-level screening.** For six countries the national
figure hides a spread wider than the spread between countries — Norway is flat
on the Oslo–Goteborg axis and mountainous to Bergen — so a routed value must
override the country value as soon as one exists.

In [ ]:
# STDLIB-ONLY cell. See the seed-export contract in calib/README.md.
import csv
from pathlib import Path


def _calib_dir() -> Path:
    """Anchor on the calib folder whatever the kernel's cwd happens to be."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "resolution.py").exists():
            return cand
    for sub in ("backend/models/infrastructure/calib", "models/infrastructure/calib"):
        if (cwd / sub).is_dir():
            return (cwd / sub).resolve()
    raise RuntimeError("cannot locate models/infrastructure/calib")


CALIB_DIR = _calib_dir()
DATA_DIR = CALIB_DIR / "route_context" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def write_data(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    """Write one committed observation table to calib/route_context/data/."""
    with open(DATA_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"route_context/data/{name}: {len(rows)} rows")


def read_data(name: str) -> list[dict]:
    with open(DATA_DIR / name, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

In [ ]:
# STDLIB-ONLY cell.
from dataclasses import dataclass, asdict
from typing import Optional

# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
NOT_LEVIED = "not_levied"  # positively documented as zero — not absent data
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"

USABLE = {SOURCED, NOT_LEVIED, DERIVED, BENCHMARK, ASSUMED}


@dataclass(frozen=True)
class SV:
    """
    One calibrated parameter with its audit trail.

    source_id points at sources_register.csv; locator is what makes it
    re-checkable a year later (section, table, sheet), because a network
    statement runs to hundreds of pages and its tariff tables move between
    editions. Values stay in native currency and price basis — conversion and
    escalation happen later and explicitly.
    """

    country_code: str
    parameter: str
    value: Optional[float]
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: Optional[int] = None
    note: str = ""
    low: Optional[float] = None  # sensitivity band, mandatory when ASSUMED
    high: Optional[float] = None

    def __post_init__(self):
        if self.status in USABLE and self.value is None:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: {self.status} needs a value"
            )
        if self.status in (SOURCED, NOT_LEVIED) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: sourced needs source_id + locator"
            )
        if self.status == ASSUMED and (
            self.low is None or self.high is None or not self.note
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: assumed needs a band and a rationale"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: derived must state its formula"
            )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]


def emit(name: str, values: list[SV]) -> None:
    write_data(name, SV_FIELDS, [asdict(v) for v in values])
    by_status: dict[str, int] = {}
    for v in values:
        by_status[v.status] = by_status.get(v.status, 0) + 1
    for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
        print(f"    {s:11} {n:4}")

## Terrain

Scores describe the main-line network a night train would use, not the
country's topography: railways follow valleys, so Belgium's Ardennes crossing
is harder than Austria's Danube corridor even though Austria is the alpine
country. No country averages the severest band; that exists at line level only.

In [ ]:
# Model bands on cumulative ascent, and their mapping onto the three values the
# current DB CHECK constraint accepts. The five-band split is the useful one for
# energy; widening the constraint is a schema question, not a calibration one.
BANDS = (
    (1.5, "T1", "Flat"),
    (4.0, "T2", "Hilly"),
    (8.0, "T3", "Hilly"),
    (15.0, "T4", "Mountainous"),
    (float("inf"), "T5", "Mountainous"),
)


def terrain_band(a: float) -> tuple[str, str]:
    return next((code, db) for hi, code, db in BANDS if a < hi)


def terrain_score(a: float) -> int:
    """Map ascent onto the 1-100 score the schema documents."""
    return max(1, min(100, round(5 * a)))


# country: (A m/km, ruling gradient per mille, what sets the level)
TERRAIN = {
    "NL": (0.6, 5, "entirely flat; only the Betuweroute ramps rise"),
    "DK": (0.9, 5, "glacial moraine, no relief of consequence"),
    "EE": (1.0, 6, "Baltic plain"),
    "LT": (1.1, 6, "Baltic plain"),
    "LV": (1.1, 6, "Baltic plain"),
    "HU": (1.3, 10, "Pannonian basin; only the Bakony and Matra edges rise"),
    "PL": (1.4, 12, "North European Plain; Carpathian foothills only in the far south"),
    "IE": (1.6, 12, "gentle rolling, main lines follow lowlands"),
    "FI": (1.8, 10, "lakeland, gently undulating"),
    "BE": (1.9, 16, "flat Flanders offset by the Ardennes on the Liege-Aachen axis"),
    "FR": (
        2.2,
        25,
        "large flat basins; Massif Central and Alpine approaches are edge cases",
    ),
    "PT": (
        2.5,
        25,
        "coastal Lisboa-Porto flat; the Beira Alta interior is far steeper",
    ),
    "SE": (2.6, 17, "flat in the south, rolling through Bergslagen and Norrland"),
    "DE": (2.8, 25, "flat north, Mittelgebirge on every north-south corridor"),
    "UK": (3.0, 13, "rolling; the WCML crosses Shap and Beattock"),
    "CZ": (
        3.2,
        20,
        "Bohemian basin ringed by hills; main lines follow the Labe and Vltava",
    ),
    "RO": (3.8, 20, "plains, but the Carpathian crossing at Predeal is severe"),
    "LU": (4.2, 16, "small network largely on the Oesling plateau"),
    "BG": (4.5, 25, "Balkan range bisects the country; Sofia sits at 550 m"),
    "ES": (
        4.8,
        20,
        "Meseta at 600-700 m — every line out of Madrid climbs to reach it",
    ),
    "IT": (5.0, 30, "Po valley flat, but the Apennines split every north-south route"),
    "SK": (5.2, 20, "Carpathian arc runs the length of the country"),
    "HR": (
        5.6,
        26,
        "Slavonian plain flat; the Dinaric crossings to Rijeka and Split are steep",
    ),
    "GR": (
        6.2,
        25,
        "mountainous throughout; Athens-Thessaloniki crosses several ranges",
    ),
    "SI": (7.0, 26, "Alps and Karst in a small network — little easy ground"),
    "NO": (
        7.5,
        21,
        "Bergensbanen reaches 1,222 m; the Oslo-Goteborg axis is flat by contrast",
    ),
    "AT": (
        8.5,
        27,
        "Alps dominate — Semmering, Tauern, Arlberg; only the Danube valley is easy",
    ),
    "CH": (
        9.5,
        27,
        "every transit crosses the Alps, though the base tunnels have cut the worst",
    ),
}
SPLIT = {"PT", "NO", "HR", "ES", "CH", "AT"}  # national figure known to mislead
DOUBLE_TRACTION_PERMILLE = 26
print(
    f"{len(TERRAIN)} countries; "
    f"{sum(1 for _, g, _ in TERRAIN.values() if g > DOUBLE_TRACTION_PERMILLE)} above the "
    f"{DOUBLE_TRACTION_PERMILLE} per mille double-traction threshold"
)

## Timetable buffer and dwell

```
buffer_pct = 4.0 + 2.5 x sqrt(u / u_EU27) + 6.0 x (1 - punctuality)
```

Utilisation enters as a square root because conflict probability rises
sub-linearly with density. Punctuality is a proxy for realised disturbance, not
an explanation: whether a country scores badly from under-padding or from poor
infrastructure, a new operator needs more margin either way.

Dwell is a **floor of 2 minutes** per commercial stop, additive and never
multiplied by the buffer — dwell is not running time. Where a stop exists for
another reason (loco or traction change, crew change, reversal, border control)
the longer operational requirement replaces it rather than adding to it.

In [ ]:
import math

BASE_PP, UTIL_COEF, DELAY_COEF, EU27_UTIL = 4.0, 2.5, 6.0, 18.67
MIN_DWELL_MIN = 2.0

# country: (utilisation k train-km per line-km, long-distance punctuality 2022)
UTIL_PUNCT = {
    "NL": (49.37, 0.880),
    "DK": (31.33, 0.851),
    "BE": (30.70, 0.878),
    "AT": (30.61, 0.814),
    "DE": (29.97, 0.536),
    "LU": (27.79, 0.757),
    "IT": (24.78, 0.669),
    "SI": (18.81, 0.386),
    "CZ": (18.11, 0.747),
    "FR": (15.70, 0.842),
    "SE": (15.53, 0.712),
    "HU": (14.31, 0.592),
    "SK": (14.20, 0.746),
    "PL": (13.97, 0.771),
    "PT": (13.36, 0.534),
    "NO": (12.39, 0.800),
    "ES": (11.74, 0.843),
    "IE": (8.67, 0.900),
    "FI": (8.23, 0.830),
    "BG": (7.85, 0.867),
    "HR": (7.28, 0.444),
    "LT": (6.35, 0.880),
    "EE": (5.81, 0.880),
    "LV": (5.52, 0.880),
    "GR": (5.31, 0.339),
    "RO": (5.23, 0.197),
    "CH": (55.00, 0.900),
    "UK": (38.00, 0.700),
}
# RMMS covers EU27 + NO only, and omits long-distance punctuality for four members.
ASSUMED_INPUT = {
    "CH": "both",
    "UK": "both",
    "IE": "punctuality",
    "EE": "punctuality",
    "LV": "punctuality",
    "LT": "punctuality",
    "NO": "punctuality",
}
HSR_ALLOWED_DEFAULT = False  # 2032 base scenario; scenario 3 flips this
print("dwell floor:", MIN_DWELL_MIN, "min/stop")

In [ ]:
VALUES: list[SV] = []
OUT: list[dict] = []

DWELL_NOTE = (
    "Minimum boarding and alighting time at every commercial stop, uniform across "
    "countries. A floor, not a value: a longer operational requirement replaces it. "
    "Additive, never multiplied by the buffer, since dwell is not running time."
)

for cc, (a, g, driver) in sorted(TERRAIN.items()):
    util, punct = UTIL_PUNCT[cc]
    buf = BASE_PP + UTIL_COEF * math.sqrt(util / EU27_UTIL) + DELAY_COEF * (1 - punct)
    band, db_cat = terrain_band(a)
    ai = ASSUMED_INPUT.get(cc)

    VALUES += [
        SV(
            cc,
            "terrain_ascent_m_per_km",
            a,
            "m/km",
            ASSUMED,
            "",
            "",
            "EUR",
            2026,
            f"Judgement from main-corridor topography: {driver}. Right in ranking and band, "
            f"roughly +/-1 m/km; not a measurement."
            + (
                " National figure hides a within-country spread wider than the between-country spread."
                if cc in SPLIT
                else ""
            ),
            low=max(0.0, a - 1.0),
            high=a + 1.0,
        ),
        SV(
            cc,
            "ruling_gradient_permille",
            float(g),
            "per mille",
            ASSUMED,
            "",
            "",
            "EUR",
            2026,
            "Typical ruling gradient on the country's main corridors, from published line profiles.",
            low=g * 0.8,
            high=g * 1.2,
        ),
        SV(
            cc,
            "network_utilisation",
            util,
            "k train-km per line-km",
            BENCHMARK if ai in (None, "punctuality") else ASSUMED,
            "RMMS-9" if ai in (None, "punctuality") else "",
            "Fig.5 x Fig.69" if ai in (None, "punctuality") else "",
            "EUR",
            2022,
            "total train-km per line-km"
            if ai in (None, "punctuality")
            else "not in RMMS; estimated from national network length and traffic volume",
            low=None if ai in (None, "punctuality") else util * 0.8,
            high=None if ai in (None, "punctuality") else util * 1.2,
        ),
        SV(
            cc,
            "punctuality_ld",
            punct,
            "share",
            BENCHMARK if ai is None else ASSUMED,
            "RMMS-9" if ai is None else "",
            "Fig.116" if ai is None else "",
            "EUR",
            2022,
            "long-distance and high-speed passenger services on time"
            if ai is None
            else "not reported to RMMS for long-distance services; national performance assumed",
            low=None if ai is None else punct - 0.10,
            high=None if ai is None else min(1.0, punct + 0.05),
        ),
        SV(
            cc,
            "timetable_buffer_pct",
            round(buf, 2),
            "%",
            DERIVED,
            "RMMS-9",
            "Fig.69 + Fig.116",
            "EUR",
            2022,
            f"{BASE_PP} + {UTIL_COEF} x sqrt({util:.2f}/{EU27_UTIL}) + {DELAY_COEF} x (1 - {punct:.3f}). "
            "The delay coefficient is the least anchored parameter; the Back-on-Track trip database "
            "is the intended verification.",
        ),
        SV(
            cc,
            "min_dwell_min",
            MIN_DWELL_MIN,
            "min/stop",
            ASSUMED,
            "",
            "",
            "EUR",
            2026,
            DWELL_NOTE,
            low=2.0,
            high=5.0,
        ),
        SV(
            cc,
            "hsr_allowed",
            float(HSR_ALLOWED_DEFAULT),
            "flag",
            ASSUMED,
            "",
            "",
            "EUR",
            2032,
            "2032 base scenario bars night trains from HSR infrastructure; scenario 3 flips this "
            "for every country in lockstep.",
            low=0.0,
            high=1.0,
        ),
    ]

    OUT.append(
        dict(
            country_code=cc,
            terrain_ascent_m_per_km=a,
            terrain_band=band,
            terrain_category_db=db_cat,
            terrain_score=terrain_score(a),
            ruling_gradient_permille=g,
            double_traction_flag=g > DOUBLE_TRACTION_PERMILLE,
            route_value_overrides=cc in SPLIT,
            network_utilisation=util,
            punctuality_ld=punct,
            timetable_buffer_pct=round(buf, 2),
            min_dwell_min=MIN_DWELL_MIN,
            hsr_allowed=HSR_ALLOWED_DEFAULT,
            inputs_assumed=ai or "",
        )
    )

assert all(4.0 <= r["timetable_buffer_pct"] <= 15.0 for r in OUT), (
    "buffer outside published practice"
)
assert {r["terrain_band"] for r in OUT} <= {"T1", "T2", "T3", "T4"}, (
    "T5 should not appear at country level"
)
write_data("route_context_summary.csv", list(OUT[0]), OUT)
emit("route_context.csv", VALUES)

In [ ]:
# Display / validation only — pandas is fine here, seed.py skips this cell.
import pandas as pd

_df = pd.DataFrame([asdict(v) for v in VALUES])
_reg = set(pd.read_csv(DATA_DIR / "sources_register.csv")["source_id"])
_unknown = set(_df.loc[_df.source_id.ne(""), "source_id"]) - _reg
assert not _unknown, f"unregistered source ids: {sorted(_unknown)}"
_bad = _df[(_df.status == "assumed") & (_df.low.isna() | _df.high.isna())]
assert _bad.empty, _bad
print(f"{len(_df)} values, {_df.source_id.ne('').sum()} with a source pointer")
pd.DataFrame(OUT).sort_values("terrain_ascent_m_per_km")